<a href="https://colab.research.google.com/github/JSJeong-me/Stable_Diffusion/blob/main/00%20LLM_grounded_guided_Diffusion_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

제공해주신 4일 교육 과정 설계안을 바탕으로, 실제 강의실에서 강사가 활용하고 수강생이 실습할 수 있는 **[중급 단계: LLM-grounded Diffusion 기반 기술 홍보 콘텐츠 생성]** 중심의 상세 강의 예제(강의 교안 및 실습 가이드)를 작성했습니다.

이 예제는 창의적으로 프롬프트를 꾸미는 'Guided' 단계를 넘어, 실제 산업 데이터(Fact)에 기반해 환각을 제어하는 **'Grounded'** 역량을 배양하는 데 초점을 맞췄습니다.

---

# [강의 교안] LLM-grounded Diffusion 기반 기술 홍보 콘텐츠 생성

### 📅 과정 개요

* **단원명:** 기술 문서 기반의 일관성 있는 홍보 스토리보드 및 프로토타입 영상 제작
* **난이도:** 중급 (AI 아키텍트, 콘텐츠 기획자 및 개발자 대상)
* **소요 시간:** 4시간 (이론 1시간 + 실습/과제 3시간)
* **핵심 키워드:** LLM-grounded Diffusion, Visual Consistency, Information Extraction, MoviePy

---

## 1. 이론 세션: Guided vs Grounded 개념 정립 (60분)

강사는 디퓨전 모델 제어의 패러다임 변화를 수강생에게 명확히 인지시킵니다.

* **LLM-guided Diffusion (프롬프트 제어 기법):**
* LLM이 사용자의 짧은 아이디어를 바탕으로 수식어나 화풍을 '창의적으로 확장'하는 방식입니다.
* **한계:** 제품의 실제 스펙이나 기술적 사실(Fact)과 무관한 화려하기만 한 허구의 이미지가 생성될 위험이 큽니다.


* **LLM-grounded Diffusion (근거 기반 생성 기법):**
* 원천 텍스트(제품 설명서, 기술 제안서 등)를 먼저 분석하여 엄격한 사실 매트릭스(Fact Matrix)를 생성하고, 이 범위를 절대 벗어나지 않도록 디퓨전 프롬프트를 제어합니다.
* **핵심 과제:** 시퀀스(장면)가 바뀌어도 주요 설비나 인터페이스의 디자인 UI가 유지되는 **시각적 일관성(Visual Consistency)** 확보가 필수적입니다.



---

## 2. 실습 세션: 파이프라인 워크플로우 구성 (120분)

### 📌 실습 시나리오: '스마트 친환경 냉각탑 (EcoTower)' 홍보 제작

수강생은 텍스트 형태의 기술 사양서를 토대로 6개 장면의 스토리보드 이미지를 뽑아내고, 이를 단일 MP4 영상으로 합성하는 파이프라인을 구축합니다.

### Step 1. Grounding Fact 추출 (LLM)

원천 문서에서 임의의 센서나 기능이 위조되지 않도록 구조화된 데이터를 정의합니다.

> **강사 제시 원천 데이터 (Technical Document):**
> "EcoTower는 냉각탑 수질을 실시간으로 감시하는 AI 기반 관리 시스템이다. 전도도, pH, ORP, 수온 센서를 분석하고 스케일과 부식 위험을 예측한다. 위험이 발생하면 운영자 대시보드에 원인과 권장 조치를 제공한다."

**LLM System Prompt (Fact Extractor):**

```text
You are a strict data auditor.
Extract operational facts from the provided text into a JSON format.
Do NOT infer, expand, or add any functions or components not explicitly mentioned in the text.

```

### Step 2. 일관성 유지를 포함한 스토리보드 프롬프트 빌더 (LLM)

추출된 JSON을 기반으로 각 장면에 들어갈 프롬프트를 생성하되, **Visual Consistency 규칙**을 강제합니다.

**LLM 이미지 생성 유도 프롬프트 구조:**

```text
[공통 제약 (Visual Consistency)]
- Style: Clean industrial technical proposal visualization, 16:9 aspect ratio.
- Palette: Blue-gray industrial color palette, teal accents.
- Anchor Object: The exact same cooling tower architecture must appear across all scenes. No corporate logos.

[장면별 묘사 예시 (Scene 3)]
- Image Prompt: Industrial cooling tower digital twin dashboard showing conductivity, pH, ORP and water-temperature sensor streams, blue and teal interface.
- Negative Prompt: low quality, blurry, human faces, text, badges, logos, fantasy style.

```

---

## 3. 코랩(Colab) 실행 소스 코드

수강생들이 코랩(T4 GPU) 환경에서 순차적으로 실행하여 스토리보드 영상까지 완성할 수 있는 엔드투엔드(End-to-End) 스크립트입니다.

In [1]:
# ==============================================================================
# 1. 필수 라이브러리 설치 및 임포트
# ==============================================================================
!pip install diffusers transformers accelerate moviepy

import os
import torch
from diffusers import AutoPipelineForText2Image, EulerDiscreteScheduler
from moviepy.editor import ImageClip, concatenate_videoclips

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================================================================
# 2. 고품질/고속 생성을 위한 Stable Diffusion 파이프라인 설정
# ==============================================================================
print("Loading Base Diffusion Model...")
model_id = "runwayml/stable-diffusion-v1-5"
pipeline = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device)

# 일관성 있는 구조 표현을 위해 효율적인 Euler 스케줄러 세팅
pipeline.scheduler = EulerDiscreteScheduler.from_config(pipeline.scheduler.config)

# ==============================================================================
# 3. LLM-grounded 결과물 모킹 (추출된 일관성 프롬프트 셋)
# ==============================================================================
# 실제 실습 시에는 LLM API 연동부로 대체됩니다.
storyboard_data = [
    {
        "scene_id": "S01",
        "duration": 5,
        "prompt": "An industrial cooling tower facility, exterior view, clean technical proposal visualization, blue-gray color palette, 16:9 composition",
        "negative_prompt": "blurry, low quality, human, text, logo"
    },
    {
        "scene_id": "S02",
        "duration": 5,
        "prompt": "Close-up of industrial cooling tower pipes with integrated sensors, clean technical proposal visualization, blue-gray color palette, 16:9 composition",
        "negative_prompt": "blurry, low quality, human, text, logo"
    },
    {
        "scene_id": "S03",
        "duration": 5,
        "prompt": "Industrial cooling tower digital twin dashboard showing conductivity, pH, ORP and water-temperature sensor streams, blue-gray and teal interface, 16:9 composition",
        "negative_prompt": "blurry, low quality, human, text, logo"
    }
]

# ==============================================================================
# 4. 이미지 생성 루프 실행
# ==============================================================================
generated_image_paths = []
os.makedirs("./storyboard_out", exist_ok=True)

print("\nStarting LLM-grounded Image Generation Loop...")
for scene in storyboard_data:
    sid = scene["scene_id"]
    print(f"Generating Image for {sid}...")

    # 시각적 고정을 위해 시드 제어
    generator = torch.Generator(device).manual_seed(1004)

    image = pipeline(
        prompt=scene["prompt"],
        negative_prompt=scene["negative_prompt"],
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    path = f"./storyboard_out/{sid}.png"
    image.save(path)
    generated_image_paths.append(path)

print("✅ All images generated successfully.")

# ==============================================================================
# 5. 생성된 이미지를 가이드에 맞춰 MP4 동영상으로 합성
# ==============================================================================
print("\nSynthesizing Video using MoviePy...")
clips = []
for i, path in enumerate(generated_image_paths):
    duration = storyboard_data[i]["duration"]
    # 이미지를 지정된 지속 시간만큼 노출하는 클립 생성
    clip = ImageClip(path).set_duration(duration).resize(width=1280)
    clips.append(clip)

# 클립 연결 후 인코딩
video = concatenate_videoclips(clips, method="compose")
video.write_videofile(
    "ecotower_tech_storyboard.mp4",
    fps=24,
    codec="libx264",
    audio_codec="aac"
)
print("✅ MP4 Video 'ecotower_tech_storyboard.mp4' exported successfully!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in 

Loading Base Diffusion Model...


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(



model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

Generating Image for S03...


  0%|          | 0/25 [00:00<?, ?it/s]

✅ All images generated successfully.

Synthesizing Video using MoviePy...
Moviepy - Building video ecotower_tech_storyboard.mp4.
Moviepy - Writing video ecotower_tech_storyboard.mp4




t: 100%|█████████▉| 359/360 [00:20<00:00, 13.62it/s, now=None]
                                                              

Moviepy - Done !
Moviepy - video ready ecotower_tech_storyboard.mp4
✅ MP4 Video 'ecotower_tech_storyboard.mp4' exported successfully!


---

## 4. 수강생 과제 및 채점 루브릭 (60분)

### 📝 미션 수행 요구사항

수강생은 제공된 도메인 중 하나(**친환경 스마트 팩토리** 또는 **스마트 물류센터 AMR 시스템**)를 택하여 다음 결과물을 아카이브 파일로 제출합니다.

1. 원천 문서에서 가공해 낸 **`Grounding_Facts.json`**
2. 일관성 규칙이 주입된 6개 장면의 **`Storyboard_Specification.json`**
3. 생성된 이미지 6장 및 최종 합성된 **30초 분량의 `.mp4` 파일**

### 📊 채점 기준표 (Rubric)

| 평가 항목 | 배점 | 우수 (Pass) 수립 기준 |
| --- | --- | --- |
| **문서 근거성 (Grounding)** | **25%** | 원천 기술 문서에 명시되지 않은 센서나 허구의 기능이 프롬프트에 포함되지 않았는가? |
| **장면 일관성 (Consistency)** | **20%** | 구조물 형태, 지배적인 컬러 팔레트가 모든 장면에서 유지되는가? |
| **스토리 구성력** | **20%** | 각 장면의 서사(시간 배정 및 묘사 구조)가 기승전결을 갖추었는가? |
| **출력 이미지·영상 품질** | **20%** | 디퓨전 파라미터가 최적화되어 뭉개짐이나 왜곡 현상이 적은가? |
| **허구 정보 억제 (Negative)** | **15%** | 텍스트 왜곡, 불필요한 사람의 개입 등 부적절한 요소가 통제되었는가? |